## Imports and Configuration

In [95]:
# Core
import pandas as pd
import numpy as np

# Dates
from datetime import datetime, timedelta

# Optional but VERY helpful later for holidays
import holidays

# Display tweaks (makes debugging way nicer)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# File path
DATA_FILE = "interval_reads_4.15.2026_30Meters.csv"  # <-- change this

# Toggle for test mode
TEST_MODE = False
TEST_METERS = 5   # number of meters to include in test
TEST_DAYS = 100     # number of days per meter

## Load Data

In [96]:
df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
df.head()

Shape: (96866, 12)


,MeterIdentifier,AccountNumber,AcctSub,AccountRate,Multiplier,LocationNumber,MeterPosition,ReadLogDate,ReadValue,ReadDate,UOM,CISCycleIdentifier
0,143072969,10993.0,1.0,103.0,1.0,13328.0,NaN,3/12/2026 0:00,0.552,3/12/2026 18:15,KWH,1.0
1,143072969,10993.0,1.0,103.0,1.0,13328.0,NaN,3/12/2026 0:00,0.383,3/12/2026 18:30,KWH,1.0
2,143072969,10993.0,1.0,103.0,1.0,13328.0,NaN,3/12/2026 0:00,1.849,3/12/2026 18:45,KWH,1.0
3,143072969,10993.0,1.0,103.0,1.0,13328.0,NaN,3/12/2026 0:00,2.042,3/12/2026 19:00,KWH,1.0
4,143072969,10993.0,1.0,103.0,1.0,13328.0,NaN,3/12/2026 0:00,2.558,3/12/2026 19:15,KWH,1.0


In [111]:
# SCRATCH CHECKING
print("Unique multipliers:", df["Multiplier"].nunique())
print(df["Multiplier"].unique())
#Unique cycles
print("Unique cycles:", df["CISCycleIdentifier"].nunique())
print(df["CISCycleIdentifier"].unique())

Unique multipliers: 4
[ nan   1. 160.  80.  40.]
Unique cycles: 1
[nan  1.]


## Basic Cleaning and Typing

In [98]:
print("Before cleaning:", df.shape)

# Convert datetime
df["ReadLogDate"] = pd.to_datetime(df["ReadLogDate"], errors="coerce")
df["ReadDate"] = pd.to_datetime(df["ReadDate"], errors="coerce")

# Numeric safety
df["ReadValue"] = pd.to_numeric(df["ReadValue"], errors="coerce")
df["Multiplier"] = pd.to_numeric(df["Multiplier"], errors="coerce")

# Drop obvious garbage rows
df = df.dropna(subset=["ReadDate", "ReadValue"])

# Sort for sanity
df = df.sort_values(["MeterIdentifier", "ReadDate"]).reset_index(drop=True)

print("After cleaning:", df.shape)

Before cleaning: (96866, 12)
After cleaning: (96866, 12)


## Sanity Check

In [99]:
print("Unique meters:", df["MeterIdentifier"].nunique())
print("Date range:", df["ReadDate"].min(), "→", df["ReadDate"].max())

# Check interval spacing (sample one meter)
sample_meter = df["MeterIdentifier"].iloc[0]
sample = df[df["MeterIdentifier"] == sample_meter].copy()

sample["delta"] = sample["ReadDate"].diff()
sample["delta_minutes"] = sample["delta"] / pd.Timedelta(minutes=1)
print(sample["delta"].value_counts().head())
print(sample["delta_minutes"].value_counts().head())

Unique meters: 30
Date range: 2026-03-06 12:15:00 → 2026-04-15 00:00:00
delta
0 days 00:15:00    2867
0 days 08:45:00       1
1 days 00:15:00       1
Name: count, dtype: int64
delta_minutes
15.0      2867
525.0        1
1455.0       1
Name: count, dtype: int64


# Create Run Subset

In [100]:
if TEST_MODE:
    # Pick a few meters
    meters = df["MeterIdentifier"].dropna().unique()[:TEST_METERS]
    test_df = df[df["MeterIdentifier"].isin(meters)].copy()

    # Limit to first N days per meter
    test_df["date"] = test_df["ReadDate"].dt.date
    
    test_df = (
        test_df.sort_values("ReadDate")
        .groupby("MeterIdentifier")
        .apply(lambda x: x[x["date"] <= (x["date"].min() + timedelta(days=TEST_DAYS))])
        .reset_index(drop=True)
    )

    df_test = test_df.drop(columns=["date"])

    print("Test shape:", df_test.shape)
else:
    df_test = df.copy()

## Base Time Columns

In [101]:
df_test["hour"] = df_test["ReadDate"].dt.hour
df_test["minute"] = df_test["ReadDate"].dt.minute
df_test["date"] = df_test["ReadDate"].dt.date
df_test["month"] = df_test["ReadDate"].dt.month
df_test["weekday"] = df_test["ReadDate"].dt.weekday  # 0=Mon, 6=Sun

# Weekend flag
df_test["is_weekend"] = df_test["weekday"] >= 5

## Flag Meter Reads Missing Multipliers and Fill with 1s

In [102]:
df_test["missing_multiplier"] = df_test["Multiplier"].isna()
df_test["Multiplier"] = df_test["Multiplier"].fillna(1)

## Calc Multiplied Usage

In [103]:
df_test["usage_kwh_calc"] = df_test["ReadValue"] * df_test["Multiplier"]
print(df_test.shape)

(96866, 20)


## Set Up Holidays

In [104]:
# US holidays (you can customize later)
us_holidays = holidays.US()

def get_holiday_set(years):
    holiday_dates = set()
    for y in years:
        for date in holidays.US(years=y).keys():
            holiday_dates.add(date)
    return holiday_dates

years_in_data = df_test["ReadLogDate"].dt.year.unique()
holiday_dates = get_holiday_set(years_in_data)

len(holiday_dates)

12

In [105]:
df_test.head(10)


,MeterIdentifier,AccountNumber,AcctSub,AccountRate,Multiplier,LocationNumber,MeterPosition,ReadLogDate,ReadValue,ReadDate,UOM,CISCycleIdentifier,hour,minute,date,month,weekday,is_weekend,missing_multiplier,usage_kwh_calc
0,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.073,2026-03-14 18:15:00,KWH,NaN,18,15,2026-03-14,3,5,True,True,0.073
1,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.056,2026-03-14 18:30:00,KWH,NaN,18,30,2026-03-14,3,5,True,True,0.056
2,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.050,2026-03-14 18:45:00,KWH,NaN,18,45,2026-03-14,3,5,True,True,0.050
3,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.075,2026-03-14 19:00:00,KWH,NaN,19,0,2026-03-14,3,5,True,True,0.075
4,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.100,2026-03-14 19:15:00,KWH,NaN,19,15,2026-03-14,3,5,True,True,0.100
5,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.068,2026-03-14 19:30:00,KWH,NaN,19,30,2026-03-14,3,5,True,True,0.068
6,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.058,2026-03-14 19:45:00,KWH,NaN,19,45,2026-03-14,3,5,True,True,0.058
7,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.058,2026-03-14 20:00:00,KWH,NaN,20,0,2026-03-14,3,5,True,True,0.058
8,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.089,2026-03-14 20:15:00,KWH,NaN,20,15,2026-03-14,3,5,True,True,0.089
9,142856004,NaN,NaN,NaN,1.0,NaN,NaN,2026-03-14,0.127,2026-03-14 20:30:00,KWH,NaN,20,30,2026-03-14,3,5,True,True,0.127


# Core Functions

In [106]:
# ON/OFF-PEAK CLASSIFICATION
def is_on_peak(row):
    dt = row["ReadDate"]
    
    hour = dt.hour
    month = dt.month
    day = dt.day
    weekday = dt.weekday()  # 0=Mon, 6=Sun
    
    # Weekend → OFF PEAK
    if weekday >= 5:
        return False
    
    # Good Friday (2026-04-03) → OFF PEAK
    if dt.date() == datetime(2026, 4, 3).date():
        return False
    
    # Jan–Mar → 6–9 AM
    if month in [1, 2, 3]:
        return 6 <= hour < 9
    
    # Apr 1–15 → 6–9 AM AND 1–6 PM
    if month == 4 and day <= 15:
        return (6 <= hour < 9) or (13 <= hour < 18)
    
    return False

In [107]:
# Test if Good Friday (Friday, normally has on-peak) is wholly off-peak

df_test["on_peak"] = df_test.apply(is_on_peak, axis=1)

df_test[
    df_test["ReadDate"].dt.date == datetime(2026, 4, 3).date()
]["on_peak"].value_counts()


on_peak
False    2880
Name: count, dtype: int64

## Generate Summary

In [108]:
summary = df_test.groupby(["MeterIdentifier", "on_peak", "Multiplier", "missing_multiplier"]).agg(usage_kwh_calc_sum=("usage_kwh_calc", "sum"), usage_kwh_raw_sum=("ReadValue", "sum")).reset_index()
print(summary)

    MeterIdentifier  on_peak  Multiplier  missing_multiplier  usage_kwh_calc_sum  usage_kwh_raw_sum
0         142856004    False         1.0                True         1839.672000        1839.672000
1         142856004     True         1.0                True           73.996000          73.996000
2         142856005    False         1.0               False         1705.039000        1705.039000
3         142856005     True         1.0               False          210.525000         210.525000
4         143072967    False         1.0                True            7.721000           7.721000
..              ...      ...         ...                 ...                 ...                ...
58        223301158     True        40.0               False          537.880000          13.447000
59        223360865    False         1.0               False         3822.439000        3822.439000
60        223360865     True         1.0               False          617.969000         617.969000


In [110]:
summary.to_csv("summary.csv", index=False)